In [48]:
import json
import re
import sqlite3
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from extract.recover_lda_client_zip_codes import (
    get_client_html_file,
    get_client_zip_codes,
)

In [10]:
conn = sqlite3.connect("E://irs990_full.db")

In [11]:
# We only care about US clients as that is outside of the scope
# Zip codes are already collected for clients that are also registrants
query = """
SELECT
    raw_json ->> '$.filing_uuid' as filing_id,
    raw_json ->> '$.filing_type_display' as filing_type,
    raw_json ->> '$.filing_document_url' as filing_doc,
    raw_json ->> '$.filing_document_content_type' as filing_doc_type,
    raw_json ->> '$.client.id' as client_id,
    raw_json ->> '$.client.general_description' as client_general_description,
    raw_json ->> '$.client.client_self_select' as client_self_select,
    raw_json ->> '$.client.state' as client_state,
    raw_json ->> '$.client.country' as client_country
FROM lda_filings
WHERE
    raw_json ->> '$.client' is not null
    and raw_json ->> '$.client.client_self_select' <> 1
    and raw_json ->> '$.client.country' in ('US', null)
"""

In [12]:
filings = pd.read_sql(
    query,
    conn
)
print(len(filings))
filings.head()

79341


,filing_id,filing_type,filing_doc,filing_doc_type,client_id,client_general_description,client_self_select,client_state,client_country
0,7866327b-c892-4430-b9f0-1f0f679c58c6,Registration,https://lda.senate.gov/filings/public/filing/7...,text/html,58116,Emergency services dispatch center,0,IL,US
1,d81e9034-9f25-4dde-be8d-6d6d251370f1,Registration,https://lda.senate.gov/filings/public/filing/d...,text/html,54165,Connectivity provider,0,DC,US
2,c6bb3242-0e97-4a95-be5b-31f7ca998f88,Registration,https://lda.senate.gov/filings/public/filing/c...,text/html,54166,State Commission representing the California a...,0,CA,US
3,8d148bc8-9865-4583-87a6-403122a638d4,Registration,https://lda.senate.gov/filings/public/filing/8...,text/html,54168,"A cement and concrete technology company, offe...",0,TX,US
4,21fe6923-4997-4d99-b9b0-8bc775f0e98a,Registration,https://lda.senate.gov/filings/public/filing/2...,text/html,54169,Medical Marijuana Dispensary,0,OH,US


In [24]:
client_zip = {}
for i in range(len(filings)):
    if i < 20_000:
        continue
    if i % 1000 == 0:
        print(f"{i:,} filings parsed")
    # if i >= 20_000:
    #     break
    row = filings.iloc[i].to_dict()
    addresses = get_client_zip_codes(
        soup=get_client_html_file(row['filing_doc'])
    )
    
    if not addresses:
        client_zip[row['filing_id']] = {
            'client': row['client_id'],
            'address_1': None,
            'address_2': None
        }
        continue
        
    add_1, add_2 = addresses
    client_zip[row['filing_id']] = {
        'client': row['client_id'],
        # Need to convert to list if storing in a json file
        'address_1': [address_part for address_part in add_1],
        'address_2': [address_part for address_part in add_2]
    }

20,000 filings parsed
21,000 filings parsed
22,000 filings parsed
23,000 filings parsed
24,000 filings parsed
25,000 filings parsed
26,000 filings parsed
27,000 filings parsed
28,000 filings parsed
29,000 filings parsed
30,000 filings parsed
31,000 filings parsed
32,000 filings parsed
33,000 filings parsed
34,000 filings parsed
35,000 filings parsed
36,000 filings parsed
37,000 filings parsed
38,000 filings parsed
39,000 filings parsed
40,000 filings parsed
41,000 filings parsed
42,000 filings parsed
43,000 filings parsed
44,000 filings parsed
45,000 filings parsed
46,000 filings parsed
47,000 filings parsed
48,000 filings parsed
49,000 filings parsed
50,000 filings parsed
51,000 filings parsed
52,000 filings parsed
53,000 filings parsed
54,000 filings parsed
55,000 filings parsed
56,000 filings parsed
57,000 filings parsed
58,000 filings parsed
59,000 filings parsed
60,000 filings parsed
61,000 filings parsed
62,000 filings parsed
63,000 filings parsed
64,000 filings parsed
65,000 fil

In [ ]:
client_zip_data = []
for uuid, client in client_zip.items():
    row = [
        uuid,
        client['client']
    ]
    
    row += [""] * 4 if len(client['address_1']) == 0 else [part for part in client['address_1']]    
    row += [""] * 4 if len(client['address_2']) == 0 else [part for part in client['address_2']]
    client_zip_data.append(row)

client_zip_df = pd.DataFrame(
    client_zip_data,
    columns=['filing_uuid', 'client_id', 'city_1', 'state_1', 'zip_1', 'country_1', 'city_2', 'state_2', 'zip_2', 'country_2']
)
client_zip_df

In [47]:
client_zip_df.to_csv(ROOT / 'data/client_zip.csv', index=False)